<a href="https://colab.research.google.com/github/banteamlak1888/Floridan-University-Fundamentals-AI-Trianing/blob/main/unsupervised_LearningWith_Hyperparameter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Module 2.5 Applied to Wine Unsupervised Project**
*  silhouette_score → replaces accuracy
*  Pipeline → ensures consistent preprocessing
*  KMeans → clustering model

In [ ]:
# ============================================================
# MODULE 2.5 — APPLIED TO WINE UNSUPERVISED PROJECT
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.metrics import silhouette_score

**Step 2: Load Dataset**

In [ ]:
wine = load_wine(as_frame=True)

data = wine.data.copy()

print(data.head())

   alcohol  malic_acid   ash  alcalinity_of_ash  magnesium  total_phenols  \
0    14.23        1.71  2.43               15.6      127.0           2.80   
1    13.20        1.78  2.14               11.2      100.0           2.65   
2    13.16        2.36  2.67               18.6      101.0           2.80   
3    14.37        1.95  2.50               16.8      113.0           3.85   
4    13.24        2.59  2.87               21.0      118.0           2.80   

   flavanoids  nonflavanoid_phenols  proanthocyanins  color_intensity   hue  \
0        3.06                  0.28             2.29             5.64  1.04   
1        2.76                  0.26             1.28             4.38  1.05   
2        3.24                  0.30             2.81             5.68  1.03   
3        3.49                  0.24             2.18             7.80  0.86   
4        2.69                  0.39             1.82             4.32  1.04   

   od280/od315_of_diluted_wines  proline  
0                  

**Step 3: Scale Data**

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(data)

**Step 4: Train Baseline Model**

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42)

clusters = kmeans.fit_predict(X_scaled)

**Step 5: Evaluate Baseline**
* measures cluster separation
* higher = better



In [ ]:
score = silhouette_score(X_scaled, clusters)

print("Baseline Silhouette Score:", score)

Baseline Silhouette Score: 0.2848589191898987


**CROSS-VALIDATION (UNSUPERVISED ADAPTATION)**
**Step 6: Custom Cross-Validation**
We simulate cross-validation:
*  split dataset into folds
*  train on part
*  evaluate on unseen data

This checks cluster stability

In [ ]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = []

for train_idx, test_idx in kf.split(X_scaled):

    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]

    model = KMeans(n_clusters=3, random_state=42)

    model.fit(X_train)

    labels = model.predict(X_test)

    score = silhouette_score(X_test, labels)

    scores.append(score)

print("Cross-validation scores:", scores)
print("Average score:", np.mean(scores))

**HYPERPARAMETER TUNING**
**Step 7: Tune Number of Clusters (K)**
This is unsupervised hyperparameter tuning:
*  instead of tuning "accuracy"
*  we tune based on silhouette score

In [ ]:
k_values = range(2, 10)

sil_scores = []

for k in k_values:

    model = KMeans(n_clusters=k, random_state=42)

    labels = model.fit_predict(X_scaled)

    score = silhouette_score(X_scaled, labels)

    sil_scores.append(score)

# Plot results
plt.plot(k_values, sil_scores, marker='o')
plt.title("Hyperparameter Tuning (K)")
plt.xlabel("Number of Clusters")
plt.ylabel("Silhouette Score")
plt.show()

**Step 8: Select Best K**

We choose the K that gives highest score

In [ ]:
best_k = k_values[np.argmax(sil_scores)]

print("Best number of clusters:", best_k)

Best number of clusters: 3


**Step 9: Train Final Model**


In [ ]:
final_model = KMeans(n_clusters=best_k, random_state=42)

final_clusters = final_model.fit_predict(X_scaled)

print("Final Silhouette Score:",
      silhouette_score(X_scaled, final_clusters))

Final Silhouette Score: 0.2848589191898987


**MACHINE LEARNING PIPELINE**

 **Step 10: Build Pipeline**

 Pipeline ensures:
*  scaling always applied
*  model always consistent

In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("kmeans", KMeans(n_clusters=best_k, random_state=42))
])

**Step 11: Train Pipeline**
Important:

* preprocessing + model = ONE object


In [ ]:
pipeline.fit(data)

clusters = pipeline.predict(data)